In [2]:
pip install mysql-connector-python

  Using cached mysql_connector_python-9.2.0-cp312-cp312-win_amd64.whl.metadata (6.2 kB)
Using cached mysql_connector_python-9.2.0-cp312-cp312-win_amd64.whl (16.1 MB)
Note: you may need to restart the kernel to use updated packages.


# hacer importaciones necesarias

In [3]:
import pandas as pd
import mysql.connector
import random

# leer archivo

In [4]:
# Leer archivo Excel
df = pd.read_excel(r"D:\prueba_tecnica_deloite\Escaneo_Prueba.xlsx")

In [5]:
df.head()

,IP,First Detected,Last Detected,CVE ID,Gid,Categoria,Riesgo,Criticidad
0,192168100101,11/30/2022 11:13:23,44816.76875,CVE-2022-4135,1,Actualizacion navegadores,Alto,Transaccional
1,192168100101,44876.917361,44816.76875,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,Actualizacion navegadores,Bajo,General
2,192168100101,44754.55,44816.76875,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,Actualizacion navegadores,Alto,General
3,192168100101,07/21/2022 09:54:29,44816.76875,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",2,Actualizacion java,Medio,General
4,192168100101,07/16/2022 21:19:12,07/28/2022 10:24:01,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",3,Actualizacion software,Bajo,General


# Simular datos requeridos como en la base de datos no tiene  impacto, probabilidad, ni riesgo

In [6]:
# Simular datos requeridos
#CVE ID (Common Vulnerabilities and Exposures)
# contiene el identificador único de vulnerabilidades reales
df['nombre'] = df['CVE ID'].astype(str).str.slice(0, 255)
df['impacto'] = [random.randint(1, 10) for _ in range(len(df))] 
# creamos aleatoriamente impactos y probabilidades
df['probabilidad'] = [random.randint(1, 10) for _ in range(len(df))]
df['riesgo'] = df['impacto'] * df['probabilidad'] #impacto * probabilidad

In [7]:
df.head()

,IP,First Detected,Last Detected,CVE ID,Gid,Categoria,Riesgo,Criticidad,nombre,impacto,probabilidad,riesgo
0,192168100101,11/30/2022 11:13:23,44816.76875,CVE-2022-4135,1,Actualizacion navegadores,Alto,Transaccional,CVE-2022-4135,10,7,70
1,192168100101,44876.917361,44816.76875,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,Actualizacion navegadores,Bajo,General,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,1,1
2,192168100101,44754.55,44816.76875,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,Actualizacion navegadores,Alto,General,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,5,5
3,192168100101,07/21/2022 09:54:29,44816.76875,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",2,Actualizacion java,Medio,General,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",8,10,80
4,192168100101,07/16/2022 21:19:12,07/28/2022 10:24:01,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",3,Actualizacion software,Bajo,General,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",8,2,16


# Conectar a MySQL con los datos y hacer consulta mysql

In [ ]:
#conecta mysql con los datos
db_connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="password",
    database="ciberseguridad"
)
cursor = db_connection.cursor()

# Crear tabla si no existe
cursor.execute("""
CREATE TABLE IF NOT EXISTS riesgos (
    id INT AUTO_INCREMENT PRIMARY KEY,
    nombre VARCHAR(255) NOT NULL,
    impacto INT NOT NULL,
    probabilidad INT NOT NULL,
    riesgo FLOAT NOT NULL
)
""")

# Insertar datos
insert_query = """
INSERT INTO riesgos (nombre, impacto, probabilidad, riesgo)
VALUES (%s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cursor.execute(insert_query, (
        row['nombre'], row['impacto'], row['probabilidad'], row['riesgo']
    ))

db_connection.commit()

# Consultar riesgos > 50
cursor.execute("SELECT * FROM riesgos WHERE riesgo > 50")
resultados = cursor.fetchall()

# Mostrar resultados
for fila in resultados:
    print(fila)

# Cerrar conexión
cursor.close()
db_connection.close()

(5, 'CVE-2022-4135', 10, 7, 70.0)
(8, 'CVE-2022-34169, CVE-2022-21541, CVE-2022-21540, CVE-2022-21549', 8, 10, 80.0)
(19, 'CVE-2022-37963, CVE-2022-37962, CVE-2022-38010', 8, 10, 80.0)
(21, 'CVE-2022-41045, CVE-2022-41039, CVE-2022-41109, CVE-2022-41100, CVE-2022-41099, CVE-2022-41098, CVE-2022-41097, CVE-2022-41095, CVE-2022-41096, CVE-2022-41093, CVE-2022-41092, CVE-2022-41090, CVE-2022-41088, CVE-2022-41086, CVE-2022-41058, CVE-2022-41057,', 8, 10, 80.0)
(35, 'CVE-2022-22709, CVE-2022-21927, CVE-2022-21926, CVE-2022-21844', 9, 8, 72.0)
(42, 'CVE-2021-31984', 9, 9, 81.0)
(45, 'CVE-2021-28465', 9, 8, 72.0)
(46, 'CVE-2021-28466, CVE-2021-28464, CVE-2021-28468', 8, 9, 72.0)
(55, 'CVE-2020-14803', 9, 8, 72.0)
(63, 'CVE-2022-26832', 10, 10, 100.0)
(83, 'CVE-2025-1414', 10, 8, 80.0)
(85, 'CVE-2025-1016, CVE-2025-1019, CVE-2025-1013, CVE-2025-1010, CVE-2025-1017, CVE-2025-1020, CVE-2025-1011, CVE-2025-1018, CVE-2025-1012, CVE-2025-1014, CVE-2025-1009', 9, 8, 72.0)
(86, 'CVE-2025-0762', 8, 9